In [20]:
from html import escape
from pathlib import Path
import re

import numpy as np
import polars as pl
import torch
from IPython.display import display
from bokeh.io import output_notebook, show
from bokeh.embed import file_html
from bokeh.resources import INLINE
from bokeh.models import CDSView, ColumnDataSource, HoverTool, IndexFilter
from bokeh.palettes import Category10, Category20
from bokeh.plotting import figure
from datasets import load_dataset
from huggingface_hub import get_token
from numpy.typing import ArrayLike
from sklearn.manifold import TSNE
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from transformers.utils import logging as transformers_logging

from ettin_guardrails.model import Embedder
from ettin_guardrails.inference import compute_embeddings

transformers_logging.set_verbosity_error()
output_notebook()


Loading BokehJS ...

In [27]:
def plot_embeddings_tsne(
    embeddings: torch.Tensor,
    texts: ArrayLike,
    perplexity: float = 30,
    title: str = "t-SNE of embeddings",
    adversarial: ArrayLike | None = None,
    cluster_labels: ArrayLike | None = None,
    output_dir: str | Path = "../outputs/embedding_space_analysis",
    save_html: bool = False,
):
    labels = None if cluster_labels is None else np.asarray(cluster_labels, dtype=str)
    if texts is None:
        raise ValueError("texts must be provided")
    texts = np.asarray(texts, dtype=str)
    if embeddings.ndim != 2:
        raise ValueError("embeddings must have shape (n_samples, n_features)")
    if texts.shape != (len(embeddings),):
        raise ValueError("texts must have one entry per embedding")
    if labels is not None and labels.shape != (len(embeddings),):
        raise ValueError("cluster_labels must have one entry per embedding")
    adversarial = (
        np.full(len(embeddings), "—") if adversarial is None
        else np.asarray(adversarial, dtype=str)
    )
    if adversarial.shape != (len(embeddings),):
        raise ValueError("adversarial must have one entry per embedding")

    coordinates = TSNE(
        n_components=2, perplexity=perplexity, metric="cosine", random_state=42
    ).fit_transform(embeddings.detach().float().cpu().numpy())
    if labels is None:
        clusters, cluster_indices = np.array(["all"]), np.zeros(len(embeddings), dtype=int)
    else:
        clusters, cluster_indices = np.unique(labels, return_inverse=True)
    palette = Category10[10] if len(clusters) <= 10 else Category20[20]
    source_data = dict(
        x=coordinates[:, 0],
        y=coordinates[:, 1],
        adversarial=adversarial,
        marker=np.where(np.isin(adversarial, ["1", "1.0", "True"]), "diamond", "circle"),
        text=texts,
    )
    if labels is not None:
        source_data["group"] = [f"group {label}" for label in labels]
    source = ColumnDataSource(source_data)

    plot = figure(
        title=title,
        width=950,
        frame_height=550,
        x_axis_label="t-SNE 1",
        y_axis_label="t-SNE 2",
        tools="pan,wheel_zoom,box_zoom,reset,save",
        active_drag="pan",
        active_scroll="wheel_zoom",
    )
    relation_colors = {
        "entailment": "#2ca02c",
        "neutral": "#7f7f7f",
        "contradiction": "#d62728",
    }
    points = []
    for index, cluster in enumerate(clusters):
        view = CDSView(filter=IndexFilter(
            indices=np.flatnonzero(cluster_indices == index).tolist()
        ))
        points.append(plot.scatter(
            x="x", y="y", source=source, view=view,
            color=relation_colors.get(str(cluster), palette[index % len(palette)]),
            marker="marker",
            size=7, alpha=0.95,
            **({"legend_label": str(cluster)} if labels is not None else {}),
        ))
    if labels is not None:
        plot.legend.title = "Group"
        plot.legend.click_policy = "hide"
    group_tooltip = "<b>@group</b><br>" if labels is not None else ""
    plot.add_tools(HoverTool(
        renderers=points,
        tooltips=(
            '<div style="max-width: 450px; white-space: pre-wrap;">'
            + group_tooltip
            + 'Adversarial: @adversarial<br>@text</div>'
        ),
    ))
    if save_html:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        filename = re.sub(r"[^\w.-]+", "_", title).strip("_") or "embeddings_tsne"
        html_path = output_dir / f"{filename}.html"
        html_path.write_text(file_html(plot, INLINE, title), encoding="utf-8")
        print(f"Saved Bokeh plot: {html_path.resolve()}")
    else:
        show(plot)
    return plot

In [30]:
holdout_data = (
    pl.read_parquet(
        "hf://datasets/allenai/wildguardmix/test/wildguard_test.parquet",
        storage_options={"token": get_token()},
    )
    .select(
        "prompt",
        "adversarial",
        "subcategory",
        pl.when(
            (pl.col("prompt_harm_label") != "harmful")
            | (pl.col("prompt_harm_label").is_null() & (pl.col("subcategory") == "benign"))
        ).then(0).otherwise(1).alias("label"),
    )
)
ood_data = load_dataset("lmsys/toxic-chat", "toxicchat0124", split="test")
# SICK test: one text per row, with a shared label for each pair.
# Load the Parquet revision to avoid legacy dataset loading scripts.
sick_data = (
    pl.DataFrame(
        load_dataset(
            "RobZamp/sick",
            revision="refs/convert/parquet",
            split="test",
        ).to_dict()
    )
    .select(
        pl.col("id").alias("label"),
        pl.col("label").alias("entailment_label"),
        "relatedness_score",
        "sentence_A",
        "sentence_B",
    )
    .unpivot(
        on=["sentence_A", "sentence_B"],
        index=["label", "entailment_label", "relatedness_score"],
        variable_name="sentence",
        value_name="text",
    )
    .sort("label", "sentence")
    .unique("text")
    .select("text", "label", "entailment_label", "relatedness_score")
    .head(1000)
)


In [11]:
# finetuned_embedder, tokenizer, config = Embedder.load(
#     "../outputs/embedder/best.pt"
# )
# finetuned_embedder_v2, tokenizer, config = Embedder.load(
#     "../outputs/embedder/benign_vs_harmful_embedder.pt"
# )
# finetuned_embedder_05, tokenizer, config = Embedder.load(
#     "../outputs/checkpoint_temp_0_5.pt"
# )
# finetuned_embedder_09, tokenizer, config = Embedder.load(
#     "../outputs/checkpoint_temp_0_9.pt"
# )
models = {
    "crasy-embedder": Embedder(pooling_source="mlm_output"),
    "default-embedder": Embedder(),
    # "finetuned-embedder_09": finetuned_embedder_09,
    # "finetuned-embedder_05": finetuned_embedder_05,
    # "finetuned-embedder-v2": finetuned_embedder_v2,
}
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/116 [00:00<?, ?it/s]

In [31]:
embedding_batch_size = 4
embedding_max_length = 1024 #config["tokenizer"]["max_length"]
holdout_embeddings = {}
ood_embeddings = {}
sick_embeddings = {}
for model_name, model in models.items():
    holdout_embeddings[model_name] = compute_embeddings(
        model,
        AutoTokenizer.from_pretrained("jhu-clsp/ettin-encoder-68m"),
        holdout_data["prompt"],
        device=device,
        max_length=embedding_max_length,
        batch_size=embedding_batch_size,
    )
    ood_embeddings[model_name] = compute_embeddings(
        model,
        AutoTokenizer.from_pretrained("jhu-clsp/ettin-encoder-68m"),
        ood_data["user_input"],
        device=device,
        max_length=embedding_max_length,
        batch_size=embedding_batch_size,
    )
    sick_embeddings[model_name] = compute_embeddings(
        model,
        AutoTokenizer.from_pretrained("jhu-clsp/ettin-encoder-68m"),
        sick_data["text"],
        device=device,
        max_length=embedding_max_length,
        batch_size=embedding_batch_size,
    )
    model.cpu()

In [32]:
# SICK labels describe the relation between sentence_A and sentence_B.
# ClassLabel order: entailment, neutral, contradiction.
sick_relation_names = {"0": "entailment", "1": "neutral", "2": "contradiction"}
sick_relations = [
    sick_relation_names.get(str(label), str(label))
    for label in sick_data["entailment_label"]
]

for name, emb in sick_embeddings.items():
    plot_embeddings_tsne(
        emb,
        sick_data["text"],
        cluster_labels=sick_relations,
        adversarial=sick_data["label"],
        title=f"SICK — {name}",
        perplexity=20,
    )
for name, emb in ood_embeddings.items():
    plot_embeddings_tsne(
        emb,
        ood_data["user_input"],
        cluster_labels=ood_data["toxicity"],
        title=f"OOD — {name}",
        perplexity=20,
    )
for name, emb in holdout_embeddings.items():
    plot_embeddings_tsne(
        emb,
        holdout_data["prompt"],
        title=f"holdout — {name}",
        perplexity=20,
        adversarial=holdout_data["adversarial"],
        cluster_labels=holdout_data["label"],
    )

In [34]:
# Mean cosine similarity within complete SICK pairs, grouped by relation.
# sick_embeddings must correspond to the current row order of sick_data.
relation_names = {"0": "entailment", "1": "neutral", "2": "contradiction"}
pair_rows = {}
for row_index, (pair_id, relation) in enumerate(
    sick_data.select("label", "entailment_label").iter_rows()
):
    relation = relation_names.get(str(relation), str(relation).lower())
    pair_rows.setdefault(pair_id, []).append((row_index, relation))

complete_pairs = {"entailment": [], "contradiction": []}
skipped_pairs = 0
for pair_id, rows in pair_rows.items():
    if len(rows) != 2:
        skipped_pairs += 1
        continue
    (index_a, relation_a), (index_b, relation_b) = rows
    if relation_a != relation_b:
        raise ValueError(f"Inconsistent relation labels for SICK pair {pair_id}")
    if relation_a in complete_pairs:
        complete_pairs[relation_a].append((index_a, index_b))

print(
    f"Complete pairs: entailment={len(complete_pairs['entailment'])}, "
    f"contradiction={len(complete_pairs['contradiction'])}. "
    f"Skipped incomplete pairs: {skipped_pairs} "
    "(unique/head may remove one sentence from a pair)."
)

summary_rows = []
for model_name, embeddings in sick_embeddings.items():
    if embeddings.ndim != 2 or embeddings.shape[0] != sick_data.height:
        raise ValueError(f"{model_name}: embeddings must match the current sick_data rows")
    embeddings = embeddings.detach().float()
    result = {"model": model_name}
    n_sentences = embeddings.shape[0]
    result["all_sentences"] = n_sentences
    result["all_sentence_pairs"] = n_sentences * (n_sentences - 1) // 2
    if n_sentences < 2:
        result["all_sentences_mean_cosine"] = None
    else:
        # Sum of off-diagonal cosine similarities, without allocating an N x N matrix.
        normalized = torch.nn.functional.normalize(embeddings.cpu().double(), dim=-1, eps=1e-8)
        vector_sum = normalized.sum(dim=0)
        off_diagonal_sum = vector_sum.square().sum() - normalized.square().sum()
        result["all_sentences_mean_cosine"] = (
            off_diagonal_sum / (n_sentences * (n_sentences - 1))
        ).item()
    for relation, pairs in complete_pairs.items():
        result[f"{relation}_pairs"] = len(pairs)
        if not pairs:
            result[f"{relation}_mean_cosine"] = None
            continue
        indices = torch.tensor(pairs, dtype=torch.long, device=embeddings.device)
        similarities = torch.nn.functional.cosine_similarity(
            embeddings[indices[:, 0]], embeddings[indices[:, 1]], dim=-1,
        )
        result[f"{relation}_mean_cosine"] = similarities.mean().item()
    entailment_mean = result["entailment_mean_cosine"]
    contradiction_mean = result["contradiction_mean_cosine"]
    result["entailment_minus_contradiction"] = (
        entailment_mean - contradiction_mean
        if entailment_mean is not None and contradiction_mean is not None else None
    )
    summary_rows.append(result)

sick_cosine_summary = pl.DataFrame(summary_rows)
display(sick_cosine_summary)


Complete pairs: entailment=30, contradiction=8. Skipped incomplete pairs: 896 (unique/head may remove one sentence from a pair).


model,all_sentences,all_sentence_pairs,all_sentences_mean_cosine,entailment_pairs,entailment_mean_cosine,contradiction_pairs,contradiction_mean_cosine,entailment_minus_contradiction
str,i64,i64,f64,i64,f64,i64,f64,f64
"""crasy-embedder""",1000,499500,0.585144,30,0.937472,8,0.868119,0.069353
"""default-embedder""",1000,499500,0.959675,30,0.992602,8,0.9869,0.005702
